In [14]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
%cd /content/SC4001-Group133

/content


In [16]:
from src.setup_cache import setup
setup()

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001


In [17]:
from pathlib import Path

base = Path("/content/drive/MyDrive/SC4001")

checkpoints = [
    base / "results/checkpoints/distillation",
    base / "data/checkpoints/distillation",
    base / "data/checkpoints/distillation_updated",
]

for folder in checkpoints:
    print("\nCheckpoint:", folder)
    print("Exists:", folder.is_dir())
    print("Adapter exists:", (folder / "adapter_model.safetensors").is_file())


Checkpoint: /content/drive/MyDrive/SC4001/results/checkpoints/distillation
Exists: True
Adapter exists: True

Checkpoint: /content/drive/MyDrive/SC4001/data/checkpoints/distillation
Exists: False
Adapter exists: False

Checkpoint: /content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated
Exists: True
Adapter exists: True


In [18]:
import yaml
from pathlib import Path

config_path = Path("configs/eval.yaml")

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/results/checkpoints/distillation"
)

assert (checkpoint / "adapter_model.safetensors").is_file(), \
    "Original Experiment 3 checkpoint not found!"

assert (checkpoint / "adapter_config.json").is_file(), \
    "Original Experiment 3 adapter configuration not found!"

with config_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["model"]["checkpoint_path"] = str(checkpoint)

cfg["gsm8k"]["split"] = "test"
cfg["gsm8k"]["max_samples"] = 100

cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Original Experiment 3 configured!")
print("Checkpoint:", checkpoint)
print("Questions:", 100)
print("Max tokens:", 600)

Original Experiment 3 configured!
Checkpoint: /content/drive/MyDrive/SC4001/results/checkpoints/distillation
Questions: 100
Max tokens: 600


In [19]:
from src.evaluation.eval_gsm8k import evaluate_gsm8k

evaluate_gsm8k(
    experiment_name="exp3_original_100traces_test100_600tokens",
    max_samples=100
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating GSM8K [exp3_original_100traces_test100_600tokens]: 100%|██████████| 100/100 [04:56<00:00,  2.97s/it]


[{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'ground_truth': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18',
  'prediction': '#### 320',
  'predicted_answer': '320',
  'is_correct': False,
  'latency_ms': 212.07557399975485,
  'token_count': 6},
 {'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
  'ground_truth': 'It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3',
  'prediction': '#### $4$\n\n**Step-by-Step Solution:**\n\n1. **Identify the quantities involved:**\n   - Let\'s denote the number of bolts of blue fib

In [20]:
import pandas as pd
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/SC4001/results")

old_path = results_dir / "exp3_original_100traces_test100_600tokens_gsm8k.csv"
new_path = results_dir / "exp3_distillation_updated_500_600tokens_gsm8k.csv"

assert old_path.is_file(), "Original results CSV not found"
assert new_path.is_file(), "Updated results CSV not found"

old = pd.read_csv(old_path)
new = pd.read_csv(new_path).head(100)

assert old["question"].tolist() == new["question"].tolist(), \
    "The evaluation questions do not match!"

print("Original 100-trace model:",
      f"{old['is_correct'].mean():.2%}")

print("Updated 522-trace model:",
      f"{new['is_correct'].mean():.2%}")

print("Same 100 GSM8K questions: Yes")

Original 100-trace model: 14.00%
Updated 522-trace model: 41.00%
Same 100 GSM8K questions: Yes


In [21]:
from pathlib import Path

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/results/checkpoints/filtered"
)

print("Checkpoint exists:", checkpoint.is_dir())
print("Adapter exists:", (checkpoint / "adapter_model.safetensors").is_file())
print("Configuration exists:", (checkpoint / "adapter_config.json").is_file())

Checkpoint exists: True
Adapter exists: True
Configuration exists: True


In [22]:
import yaml
from pathlib import Path

config_path = Path("configs/eval.yaml")

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/results/checkpoints/filtered"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

with config_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["model"]["checkpoint_path"] = str(checkpoint)

cfg["gsm8k"]["split"] = "test"
cfg["gsm8k"]["max_samples"] = 100

cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Original Experiment 4 configured!")
print("Checkpoint:", checkpoint)
print("Questions:", 100)
print("Max tokens:", 600)

Original Experiment 4 configured!
Checkpoint: /content/drive/MyDrive/SC4001/results/checkpoints/filtered
Questions: 100
Max tokens: 600


In [23]:
from src.evaluation.eval_gsm8k import evaluate_gsm8k

evaluate_gsm8k(
    experiment_name="exp4_original_100traces_test100_600tokens",
    max_samples=100
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating GSM8K [exp4_original_100traces_test100_600tokens]: 100%|██████████| 100/100 [08:38<00:00,  5.18s/it]


[{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'ground_truth': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18',
  'prediction': '#### 320',
  'predicted_answer': '320',
  'is_correct': False,
  'latency_ms': 221.23403599835,
  'token_count': 6},
 {'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
  'ground_truth': 'It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3',
  'prediction': '#### 4',
  'predicted_answer': '4',
  'is_correct': False,
  'latency_ms': 145.30887000000803,
  'token_count': 4},
 {'question': 'Josh d

In [24]:
import pandas as pd
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/SC4001/results")

old_path = (
    results_dir
    / "exp4_original_100traces_test100_600tokens_gsm8k.csv"
)

updated_path = (
    results_dir
    / "exp4_filtered_updated_500_600tokens_gsm8k.csv"
)

assert old_path.is_file(), "Original Exp 4 results not found"
assert updated_path.is_file(), "Updated Exp 4 results not found"

old = pd.read_csv(old_path)
updated = pd.read_csv(updated_path).head(100)

assert len(old) == len(updated) == 100
assert old["question"].tolist() == updated["question"].tolist(), \
    "Evaluation questions do not match!"

print("Original 100-trace filtered model:",
      f"{old['is_correct'].mean():.2%}")

print("Updated filtered model:",
      f"{updated['is_correct'].mean():.2%}")

print("Same 100 GSM8K questions: Yes")

Original 100-trace filtered model: 15.00%
Updated filtered model: 34.00%
Same 100 GSM8K questions: Yes


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import sys

repo_path = "/content/SC4001-Group133"

if not os.path.exists(repo_path + "/.git"):
    !git clone https://github.com/TIFFMUN/SC4001-Group133.git /content/SC4001-Group133

os.chdir(repo_path)

!pip install -r requirements.txt -q
!pip install -e . -q

print("Project loaded!")
print("Working directory:", os.getcwd())

Cloning into '/content/SC4001-Group133'...
remote: Enumerating objects: 336, done.
remote: Total 336 (delta 0), reused 0 (delta 0), pack-reused 336 (from 1)
Receiving objects: 100% (336/336), 4.66 MiB | 3.33 MiB/s, done.
Resolving deltas: 100% (188/188), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sc4001-group133 (pyproject.toml) ... done
Project loaded!
Working directory: /content/SC4001-Group133


In [3]:
from src.setup_cache import setup

setup()

print("Storage setup completed!")

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001
Storage setup completed!


Ep 3

In [ ]:
# Experiment 3 — Standard Reasoning Distillation
# Updated dataset: 469 training examples
# Evaluation: GSM8K test, 500 questions, 600 max tokens

import yaml
from pathlib import Path
from src.evaluation.eval_gsm8k import evaluate_gsm8k

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

cfg["model"]["checkpoint_path"] = str(checkpoint)
cfg["gsm8k"]["split"] = "test"
cfg["gsm8k"]["max_samples"] = 500
cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

EXPERIMENT_NAME = "exp3_distillation_updated_500_600tokens"

print("Experiment 3 configured successfully!")
print("Checkpoint:", checkpoint)

In [ ]:
# Experiment 3 — Full GSM8K evaluation
# ALREADY COMPLETED — Do not rerun

evaluate_gsm8k(
    experiment_name="exp3_distillation_updated_500_600tokens",
    max_samples=500
)

EXP 4

In [4]:
# Experiment 4 — Filtered Distillation
# GSM8K: 500 questions, 600 max tokens

import yaml
from pathlib import Path
from src.evaluation.eval_gsm8k import evaluate_gsm8k

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

cfg["model"]["checkpoint_path"] = str(checkpoint)
cfg["gsm8k"]["split"] = "test"
cfg["gsm8k"]["max_samples"] = 500
cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

EXPERIMENT_NAME = "exp4_filtered_updated_500_600tokens"

print("Experiment 4 configured!")
print("Checkpoint:", checkpoint)
print("Questions:", 500)
print("Max tokens:", 600)

W1115 22:56:17.653000 516 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 22:56:17.716000 516 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Experiment 4 configured!
Checkpoint: /content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated
Questions: 500
Max tokens: 600


In [5]:
evaluate_gsm8k(
    experiment_name="exp4_filtered_updated_500_600tokens",
    max_samples=500
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating GSM8K [exp4_filtered_updated_500_600tokens]: 100%|██████████| 500/500 [1:35:48<00:00, 11.50s/it]


[{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'ground_truth': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18',
  'prediction': "To find out how much money Janet makes every day from selling the eggs, we need to calculate the total number of eggs laid and then multiply that by the price per egg.\n\n**Step 1: Calculate the total number of eggs laid in a day.**\n*   **Ducks:** The problem states that the ducks lay 16 eggs per day.\n*   **Breakfast:** Janet eats 3 eggs (one for breakfast) each morning.\n*   **Baking:** She bakes muffins for her friends every day, which means she also bakes muffins. Let's assume she bakes muffins for 7 days (since the probl

EXP 5 setup

In [8]:
# Experiment 5 — Cross-task transfer
# ARC-Challenge official test split

import yaml
from pathlib import Path
from src.evaluation.eval_arc import evaluate_arc

config_path = Path("configs/eval.yaml")

with config_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["arc"]["dataset"] = "allenai/ai2_arc"
cfg["arc"]["config"] = "ARC-Challenge"
cfg["arc"]["split"] = "test"
cfg["arc"]["max_samples"] = None

cfg["generation"]["max_new_tokens"] = 600
cfg["generation"]["do_sample"] = False

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("ARC-Challenge configured!")

ARC-Challenge configured!


EXp 5A

In [9]:
# Exp 3 — Standard Distillation on ARC-Challenge

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/distillation_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

with config_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["model"]["checkpoint_path"] = str(checkpoint)

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Running Exp 3 on ARC-Challenge...")

evaluate_arc(
    experiment_name="exp5_arc_exp3_updated_600tokens"
)

Running Exp 3 on ARC-Challenge...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating ARC [exp5_arc_exp3_updated_600tokens]: 100%|██████████| 1172/1172 [02:37<00:00,  7.43it/s]


[{'question': 'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?',
  'choices': ['Planetary density will decrease.',
   'Planetary years will become longer.',
   'Planetary days will become shorter.',
   'Planetary gravity will become stronger.'],
  'answer_key': 'C',
  'predicted_label': 'D',
  'is_correct': False,
  'scores': [-3.501830577850342,
   -3.47575306892395,
   -3.4102213382720947,
   -3.3922808170318604],
  'latency_ms': 126.50079299965,
  'token_count': 27},
 {'question': 'A group of engineers wanted to know how different building designs would respond during an earthquake. They made several models of buildings and tested each for its ability to withstand earthquake conditions. Which will most likely result from testing different building designs?',
  'choices': ['buildings will be built faster',
   'buildings will be made safer',
   'building designs will look nicer',
   'building m

EXP 5B

In [10]:
# Exp 4 — Filtered Distillation on ARC-Challenge

checkpoint = Path(
    "/content/drive/MyDrive/SC4001/data/checkpoints/filtered_updated"
)

assert (checkpoint / "adapter_model.safetensors").is_file()
assert (checkpoint / "adapter_config.json").is_file()

with config_path.open() as f:
    cfg = yaml.safe_load(f)

cfg["model"]["checkpoint_path"] = str(checkpoint)

with config_path.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Running Exp 4 on ARC-Challenge...")

evaluate_arc(
    experiment_name="exp5_arc_exp4_updated_600tokens"
)

Running Exp 4 on ARC-Challenge...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating ARC [exp5_arc_exp4_updated_600tokens]: 100%|██████████| 1172/1172 [02:38<00:00,  7.41it/s]


[{'question': 'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?',
  'choices': ['Planetary density will decrease.',
   'Planetary years will become longer.',
   'Planetary days will become shorter.',
   'Planetary gravity will become stronger.'],
  'answer_key': 'C',
  'predicted_label': 'D',
  'is_correct': False,
  'scores': [-3.4666409492492676,
   -3.4411211013793945,
   -3.382532835006714,
   -3.3627235889434814],
  'latency_ms': 121.91387200073223,
  'token_count': 27},
 {'question': 'A group of engineers wanted to know how different building designs would respond during an earthquake. They made several models of buildings and tested each for its ability to withstand earthquake conditions. Which will most likely result from testing different building designs?',
  'choices': ['buildings will be built faster',
   'buildings will be made safer',
   'building designs will look nicer',
   'build

In [11]:
from pathlib import Path

base = Path("/content/drive/MyDrive/SC4001")

for folder in [base / "results", base / "data" / "results"]:
    if folder.exists():
        print("\nResults in:", folder)

        for file in sorted(folder.rglob("*")):
            if file.is_file() and file.suffix in [".csv", ".json"]:
                if "exp3" in file.name or "exp4" in file.name or "exp5" in file.name:
                    print(file)


Results in: /content/drive/MyDrive/SC4001/results
/content/drive/MyDrive/SC4001/results/backup_local_results_20261113_101250/exp5_d3_cross_task_arc.csv
/content/drive/MyDrive/SC4001/results/exp3_distillation_gsm8k.csv
/content/drive/MyDrive/SC4001/results/exp3_distillation_updated_500_600tokens_gsm8k.csv
/content/drive/MyDrive/SC4001/results/exp4_filtered_updated_500_600tokens_gsm8k.csv
/content/drive/MyDrive/SC4001/results/exp5_arc_exp3_updated_600tokens_arc.csv
/content/drive/MyDrive/SC4001/results/exp5_arc_exp4_updated_600tokens_arc.csv
/content/drive/MyDrive/SC4001/results/exp5_d3_cross_task_arc.csv
/content/drive/MyDrive/SC4001/results/summary_results/exp3_distillation_gsm8k_summary.json
/content/drive/MyDrive/SC4001/results/summary_results/exp3_distillation_updated_500_600tokens_gsm8k_summary.json
/content/drive/MyDrive/SC4001/results/summary_results/exp4_filtered_updated_500_600tokens_gsm8k_summary.json


In [12]:
from pathlib import Path

base = Path("/content/drive/MyDrive/SC4001")

for name in ["exp5_arc_exp3_updated_600tokens",
             "exp5_arc_exp4_updated_600tokens"]:

    print(f"\n{name}")

    matches = [
        p for p in base.rglob("*")
        if p.is_file() and name in p.name
    ]

    if matches:
        for p in matches:
            print("Saved:", p)
    else:
        print("No saved files found")


exp5_arc_exp3_updated_600tokens
Saved: /content/drive/MyDrive/SC4001/results/exp5_arc_exp3_updated_600tokens_arc.csv

exp5_arc_exp4_updated_600tokens
Saved: /content/drive/MyDrive/SC4001/results/exp5_arc_exp4_updated_600tokens_arc.csv


In [13]:
import pandas as pd
import json
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/SC4001/results")

experiments = [
    "exp5_arc_exp3_updated_600tokens",
    "exp5_arc_exp4_updated_600tokens"
]

for name in experiments:
    csv_path = results_dir / f"{name}_arc.csv"
    json_path = results_dir / f"{name}_arc_summary.json"

    df = pd.read_csv(csv_path)

    print(f"\nExperiment: {name}")
    print("CSV columns:", df.columns.tolist())
    print("Total questions:", len(df))

    # Calculate evaluation metrics
    summary = {
        "experiment": name,
        "accuracy": float(df["is_correct"].mean()),
        "mean_latency_ms": float(df["latency_ms"].mean()),
        "median_latency_ms": float(df["latency_ms"].median()),
        "mean_token_count": float(df["token_count"].mean()),
        "n_samples": len(df)
    }

    # Save JSON summary
    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\nSummary:")
    print(json.dumps(summary, indent=2))
    print("\nSaved:", json_path)


Experiment: exp5_arc_exp3_updated_600tokens
CSV columns: ['question', 'choices', 'answer_key', 'predicted_label', 'is_correct', 'scores', 'latency_ms', 'token_count']
Total questions: 1172

Summary:
{
  "experiment": "exp5_arc_exp3_updated_600tokens",
  "accuracy": 0.3361774744027304,
  "mean_latency_ms": 132.41387373721412,
  "median_latency_ms": 123.99093549993268,
  "mean_token_count": 25.04778156996587,
  "n_samples": 1172
}

Saved: /content/drive/MyDrive/SC4001/results/exp5_arc_exp3_updated_600tokens_arc_summary.json

Experiment: exp5_arc_exp4_updated_600tokens
CSV columns: ['question', 'choices', 'answer_key', 'predicted_label', 'is_correct', 'scores', 'latency_ms', 'token_count']
Total questions: 1172

Summary:
{
  "experiment": "exp5_arc_exp4_updated_600tokens",
  "accuracy": 0.3438566552901024,
  "mean_latency_ms": 132.76691017487073,
  "median_latency_ms": 123.90998049977496,
  "mean_token_count": 25.04778156996587,
  "n_samples": 1172
}

Saved: /content/drive/MyDrive/SC4001